<h1 style="color:DodgerBlue">Индивидальный проект</h1>

<h2 style="color:DodgerBlue">Название проекта:</h2>

----

### Вариант задания № 17


<h2 style="color:DodgerBlue">Описание проекта:</h2>

----

Развитие системы расчета опций доставки логистической компании. В данной версии проект демонстрирует расширенное применение принципов ООП:

1. **Полная инкапсуляция:** Вместо автосвойств добавлены закрытые (private) поля. Доступ к ним осуществляется через развернутые геттеры и сеттеры с логикой валидации (проверка на отрицательные значения, пустые строки и т.д.).
2. **Безопасные конструкторы:** Конструкторы инициализируют объекты не через прямую запись в поля, а через сеттеры свойств, чтобы правила валидации применялись уже в момент создания объекта.
3. **Взаимодействие объектов:** Добавлен класс `Order` (Заказ). Теперь объекты опций доставки (`ShippingOption`) и заказов (`Order`) взаимодействуют друг с другом. Метод расчета стоимости доставки принимает объект заказа в качестве параметра и динамически меняет цену в зависимости от свойств конкретного заказа (например, веса посылки).

#### Дополнительное задание
Добавьте к сущестующим классам конструктора классов с использованием гетторов и сетторов и реализуйте взаимодействие объектов между собой

<h2 style="color:DodgerBlue">Реализация:</h2>

----

In [1]:
using System;

// === КЛАСС ДЛЯ ДЕМОНСТРАЦИИ ВЗАИМОДЕЙСТВИЯ ===
public class Order
{
    private int _orderId;
    private double _packageWeight;

    public int OrderId
    {
        get { return _orderId; }
        set { _orderId = value > 0 ? value : throw new ArgumentException("ID заказа должен быть больше 0"); }
    }

    public double PackageWeight
    {
        get { return _packageWeight; }
        set { _packageWeight = value > 0 ? value : throw new ArgumentException("Вес должен быть положительным"); }
    }

    // Конструктор использует сеттеры для встроенной проверки данных
    public Order(int orderId, double packageWeight)
    {
        OrderId = orderId;
        PackageWeight = packageWeight;
    }

    // Взаимодействие объектов: Заказ принимает объект доставки для обработки
    public void Checkout(ShippingOption shippingOption)
    {
        Console.WriteLine($"\n--- Обработка заказа #{OrderId} (Вес: {PackageWeight} кг) ---");
        Console.WriteLine(shippingOption.GetDeliveryDetails());
        
        // Объект доставки вычисляет цену на основе данных этого заказа
        decimal finalCost = shippingOption.CalculateCost(this);
        
        Console.WriteLine($"Сроки: {shippingOption.EstimateDeliveryTime()}");
        Console.WriteLine($"Итоговая стоимость доставки: {finalCost:C}");
    }
}

// === БАЗОВЫЙ КЛАСС ===
public class ShippingOption
{
    // Закрытые поля для инкапсуляции
    private int _deliveryOptionId;
    private string _deliveryOptionName;
    private decimal _baseCost;

    // Свойства с геттерами и сеттерами (с валидацией)
    public int DeliveryOptionId
    {
        get { return _deliveryOptionId; }
        set { _deliveryOptionId = value; }
    }

    public string DeliveryOptionName
    {
        get { return _deliveryOptionName; }
        set { _deliveryOptionName = string.IsNullOrWhiteSpace(value) ? "Неизвестный способ" : value; }
    }

    public decimal BaseCost
    {
        get { return _baseCost; }
        set { _baseCost = value >= 0 ? value : 0; }
    }

    // Конструктор использует сеттеры свойств для безопасной инициализации
    public ShippingOption(int id, string name, decimal baseCost)
    {
        DeliveryOptionId = id;
        DeliveryOptionName = name;
        BaseCost = baseCost;
    }

    // Взаимодействие: метод принимает объект Order и использует его данные
    public virtual decimal CalculateCost(Order order)
    {
        // Логика: базовая стоимость + наценка 50 руб. за каждый килограмм свыше 1 кг
        decimal weightSurcharge = order.PackageWeight > 1.0 ? (decimal)(order.PackageWeight - 1.0) * 50m : 0m;
        return BaseCost + weightSurcharge;
    }

    public virtual string EstimateDeliveryTime()
    {
        return "Время доставки не определено.";
    }

    public virtual string GetDeliveryDetails()
    {
        return $"[ID: {DeliveryOptionId}] Способ: {DeliveryOptionName} | Базовая цена: {BaseCost:C}";
    }
}

// === ПРОИЗВОДНЫЙ КЛАСС 1: СТАНДАРТНАЯ ДОСТАВКА ===
public class StandardDelivery : ShippingOption
{
    private int _averageDeliveryTime;

    public int AverageDeliveryTime
    {
        get { return _averageDeliveryTime; }
        set { _averageDeliveryTime = value > 0 ? value : 1; }
    }

    public StandardDelivery(int id, string name, decimal baseCost, int averageDeliveryTime) 
        : base(id, name, baseCost)
    {
        AverageDeliveryTime = averageDeliveryTime;
    }

    public override string EstimateDeliveryTime()
    {
        return $"{AverageDeliveryTime} рабочих дней.";
    }
}

// === ПРОИЗВОДНЫЙ КЛАСС 2: ЭКСПРЕСС-ДОСТАВКА ===
public class ExpressDelivery : ShippingOption
{
    private int _minDeliveryTime;
    private decimal _expressSurcharge;

    public int MinDeliveryTime
    {
        get { return _minDeliveryTime; }
        set { _minDeliveryTime = value > 0 ? value : 1; }
    }

    public decimal ExpressSurcharge
    {
        get { return _expressSurcharge; }
        set { _expressSurcharge = value >= 0 ? value : 0; }
    }

    public ExpressDelivery(int id, string name, decimal baseCost, int minDeliveryTime, decimal expressSurcharge) 
        : base(id, name, baseCost)
    {
        MinDeliveryTime = minDeliveryTime;
        ExpressSurcharge = expressSurcharge;
    }

    // Полиморфизм + взаимодействие
    public override decimal CalculateCost(Order order)
    {
        // Считаем цену от веса заказа (базовый метод) и добавляем наценку за срочность
        return base.CalculateCost(order) + ExpressSurcharge;
    }

    public override string EstimateDeliveryTime()
    {
        return $"Срочная доставка (до {MinDeliveryTime} часов).";
    }
}

// === ПРОИЗВОДНЫЙ КЛАСС 3: САМОВЫВОЗ ===
public class Pickup : ShippingOption
{
    private string _pickupAddress;

    public string PickupAddress
    {
        get { return _pickupAddress; }
        set { _pickupAddress = string.IsNullOrWhiteSpace(value) ? "Адрес не указан" : value; }
    }

    public Pickup(int id, string name, decimal baseCost, string pickupAddress) 
        : base(id, name, baseCost)
    {
        PickupAddress = pickupAddress;
    }

    public override decimal CalculateCost(Order order)
    {
        // Вес заказа не влияет на стоимость при самовывозе
        return BaseCost;
    }

    public override string EstimateDeliveryTime()
    {
        return "Уже в пункте выдачи. Можно забирать.";
    }

    public override string GetDeliveryDetails()
    {
        return $"{base.GetDeliveryDetails()} | Адрес ПВЗ: {PickupAddress}";
    }
}

// =========================================
// ДЕМОНСТРАЦИЯ ИСПОЛНЕНИЯ
// =========================================

// Создаем объекты заказов с разным весом
Order orderLight = new Order(101, 0.5); // Легкий заказ (0.5 кг)
Order orderHeavy = new Order(102, 5.0); // Тяжелый заказ (5.0 кг)

// Создаем объекты опций доставки
StandardDelivery stdDelivery = new StandardDelivery(1, "Почта России", 200m, 5);
ExpressDelivery expDelivery = new ExpressDelivery(2, "Курьер EMS", 400m, 24, 350m);
Pickup pickup = new Pickup(3, "ПВЗ СДЭК", 150m, "г. Москва, ул. Пушкина, д. 10");

// Демонстрация взаимодействия объектов
// Заказы "общаются" с объектами доставки для расчета финальной суммы

orderLight.Checkout(stdDelivery); // Стандартная цена без доплат за вес
orderHeavy.Checkout(stdDelivery); // +200 руб доплаты за лишние 4 кг веса

orderHeavy.Checkout(expDelivery); // Доплата за вес + экспресс-наценка

orderHeavy.Checkout(pickup);      // Вес игнорируется при самовывозе, стоимость фиксирована